# Causal Inference: Early Feature Adoption and 90-Day Account Conversion

## Director-Level Project Brief

This notebook answers a product strategy question that pure predictive ML cannot answer:

> Which early product behaviors appear to **causally improve** account conversion, for which accounts, and what should we change in the customer experience?

The best first project is:

```text
Estimate the causal impact of early feature adoption on 90-day account conversion,
with heterogeneous treatment effects and recommended interventions.
```

This is different from the ML scoring notebook. The ML model asks:

```text
Which accounts are likely to convert?
```

This causal notebook asks:

```text
If we help similar accounts adopt a feature earlier, do they convert more often?
```

That distinction matters. Predictive models rank accounts. Causal inference helps Product, Growth, Sales, and CS decide what to change.

## 1. Executive Summary of The Approach

We will estimate the effect of early feature adoption on future account conversion.

### Unit of Analysis

`account_id`

### Time Zero

The account's first known product login:

```text
account_first_login_at = min(first_logged_in_at across users in account)
```

### Treatments

We test several product adoption treatments:

1. `workspace_created` within 7 days of first account login
2. `report_generated` within 7 days of first account login
3. `workflow_created` within 14 days of first account login
4. `api_call_made` within 14 days of first account login

### Outcome

```text
1 if account creates a Won / Closed Won deal within 90 days after treatment window closes
0 otherwise
```

For example, if treatment is `workspace_created` within 7 days, the outcome window is days 8-97 after first login.

This avoids measuring treatment and outcome in the same time window.

### Estimators

For each treatment, we estimate:

- Naive difference in conversion rates
- Propensity-score weighted ATE
- Doubly robust AIPW ATE
- Heterogeneous treatment effects by segment, industry, and account profile
- Prioritized intervention opportunities

### Why This Is Senior-Level

This notebook explicitly handles:

- Time ordering
- Leakage prevention
- Confounding
- Positivity / overlap diagnostics
- Doubly robust estimation
- Confidence intervals via bootstrap
- Heterogeneous treatment effects
- Business translation into interventions

## 2. Causal Question and Assumptions

### Core Causal Question

For accounts that are similar in observed pre-treatment characteristics, does early adoption of a specific feature increase the probability of conversion within 90 days?

Formally:

```text
ATE = E[Y(1) - Y(0)]
```

Where:

- `Y(1)` is the potential conversion outcome if the account adopts the feature early
- `Y(0)` is the potential conversion outcome if the account does not adopt the feature early

### Key Assumptions

1. **Consistency**  
   The observed outcome equals the potential outcome under the treatment actually received.

2. **Conditional Exchangeability / No Unobserved Confounding**  
   After controlling for account profile, user profile, marketing source, geography, and pre-treatment sales activity, treated and untreated accounts are comparable.

3. **Positivity / Overlap**  
   Every type of account has some chance of receiving and not receiving treatment.

4. **No Post-Treatment Controls**  
   We do not control for behavior that occurs after the treatment window begins, because it may be caused by the treatment.

### Practical Limitation

`deals_raw.csv` has `created_date`, not true close date or stage transition date. We use `created_date` as a proxy for opportunity/conversion timing. In production, replace this with true stage transition timestamps.

## 3. Conceptual DAG

A simplified causal graph:

```text
Account fit ───────────────► Early feature adoption ─────────────► Conversion
     │                              ▲                                  ▲
     │                              │                                  │
     ├──────────────► Sales activity ┘                                  │
     │                                                                 │
     ├──────────────► Marketing source ────────────────────────────────┤
     │                                                                 │
     └──────────────► User role / experience / geography ──────────────┘
```

### Confounders We Control For

- Account segment
- Industry
- Estimated annual recurring revenue
- Number of users known at baseline
- User role / job title composition
- Experience level
- Marketing source and campaign before first login
- Lead cost before first login
- Sales activity before first login
- Geography before first login
- Prior deals before first login
- Calendar cohort

### What We Avoid Controlling For

- Product events after first login, except the treatment itself
- Sales activity after treatment starts
- Deal stage after treatment starts
- Any future behavior that could be caused by early feature adoption

In [1]:
# If needed in your local notebook environment:
# %pip install pandas numpy scikit-learn matplotlib seaborn

from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
SEED_DIR = Path("lightdash-demo-saas/seeds")
WON_STAGES = {"won", "closed won"}
OUTCOME_HORIZON_DAYS = 90
CROSS_FIT_REPEATS = 5
CONFIDENCE_LEVEL = 0.95

## 4. Load and Prepare Data

We use the local seed files. This notebook is designed to run without Snowflake so causal analysis can be developed locally and later productionized.

In [2]:
from pathlib import Path
import os

import pandas as pd
import snowflake.connector


def load_env(path: str = ".env") -> None:
    env_path = Path(path)

    if not env_path.exists():
        raise FileNotFoundError(f"Could not find {path}")

    for line in env_path.read_text().splitlines():
        line = line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(
            key.strip(),
            value.strip().strip('"').strip("'")
        )


load_env()

In [3]:
private_key_file='/path/to/rsa_key.p8'
connection_params = {
    "private_key_file": "/Users/naghmeh6/agentic_ai/ABtesting/rsa_key.p8",
    "private_key_file": "/Users/naghmeh6/agentic_ai/ABtesting/rsa_key.p8",
    "authenticator": os.environ["SNOWFLAKE_AUTHENTICATOR"],
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USERNAME"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "warehouse": os.environ["SNOWFLAKE_WAREHOUSE"],
    "database": os.environ["SNOWFLAKE_DATABASE"],
    "schema": os.environ["SNOWFLAKE_SCHEMA"],
}

connection_params = {
    key: value
    for key, value in connection_params.items()
    if value is not None
}

def read_snowflake_table(table_name: str, date_cols: list[str] | None = None) -> pd.DataFrame:
    query = f"""
        SELECT *
        FROM {table_name}
    """

    with snowflake.connector.connect(**connection_params) as conn:
        df = pd.read_sql(query, conn)
    if not df.empty and df.columns[:1].values[0] == "C1":
        # Set the first row as the column headers
        df.columns = df.iloc[0].astype(str)
        # Drop the first row since it is now the header
        df = df[1:].reset_index(drop=True)
    
    df.columns = df.columns.str.lower()
    for col in date_cols or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df
    


accounts = read_snowflake_table("ACCOUNTS_RAW")
users = read_snowflake_table("USERS_RAW",["created_at", "first_logged_in_at", "latest_logged_in_at"])
tracks = read_snowflake_table("TRACKS_RAW",["event_timestamp"])
deals = read_snowflake_table("DEALS_RAW",["created_date"])
user_deals = read_snowflake_table("USER_DEALS_RAW")

marketing_leads = read_snowflake_table("MARKETING_LEADS",["created_at", "converted_at"])
activities = read_snowflake_table("ACTIVITIES_RAW",["activity_timestamp"])
lead_geo = read_snowflake_table("LEAD_GEOGRAPHIC_DATA")
addresses = read_snowflake_table("ADDRESSES_RAW",["valid_from", "valid_to"])

In [4]:
accounts["estimated_annual_recurring_revenue"] = pd.to_numeric(
    accounts["estimated_annual_recurring_revenue"], errors="coerce"
)
users["is_marketing_opted_in"] = pd.to_numeric(
    users["is_marketing_opted_in"], errors="coerce"
).fillna(0)
users["experience_in_years"] = pd.to_numeric(
    users["experience_in_years"], errors="coerce"
)
deals["amount"] = pd.to_numeric(deals["amount"], errors="coerce").fillna(0)
deals["seats"] = pd.to_numeric(deals["seats"], errors="coerce")
deals["is_won"] = deals["stage"].str.lower().isin(WON_STAGES).astype(int)
marketing_leads["lead_cost"] = pd.to_numeric(
    marketing_leads["lead_cost"], errors="coerce"
).fillna(0)
activities["call_duration_seconds"] = pd.to_numeric(
    activities["call_duration_seconds"], errors="coerce"
)

activities["lead_id"] = activities["lead_id"].astype(str)
lead_geo["lead_id"] = lead_geo["lead_id"].astype(str)
marketing_leads["user_id"] = marketing_leads["user_id"].astype(str)

tracks = tracks.merge(users[["user_id", "account_id"]], on="user_id", how="left")
marketing_leads = marketing_leads.merge(
    users[["user_id", "account_id"]], on="user_id", how="left"
)
activities = activities.merge(
    marketing_leads[["lead_id", "user_id", "account_id"]],
    on="lead_id",
    how="left",
)
lead_geo = lead_geo.merge(
    marketing_leads[["lead_id", "account_id", "created_at"]],
    on="lead_id",
    how="left",
)

print("accounts", accounts.shape)
print("users", users.shape)
print("tracks", tracks.shape)
print("deals", deals.shape)
print("marketing_leads", marketing_leads.shape)
print("activities", activities.shape)
print("won deal rate", deals["is_won"].mean().round(3))


accounts (1005, 5)
users (6006, 9)
tracks (100007, 5)
deals (1084, 8)
marketing_leads (2868, 14)
activities (14925, 10)
won deal rate 0.33


## 5. Build Account-Level Baseline Features

Baseline features must be known before or at `account_first_login_at`. This is where many causal analyses accidentally leak future information. We are intentionally conservative.

In [5]:
def mode_or_unknown(series: pd.Series) -> str:
    series = series.dropna()
    if series.empty:
        return "Unknown"
    return str(series.mode().iloc[0])


def safe_divide(numerator: float, denominator: float) -> float:
    if denominator == 0 or pd.isna(denominator):
        return 0.0
    return float(numerator / denominator)


account_first_login = (
    users.dropna(subset=["first_logged_in_at"])
    .groupby("account_id", as_index=False)
    .agg(account_first_login_at=("first_logged_in_at", "min"))
)

base_accounts = accounts.merge(account_first_login, on="account_id", how="inner")
base_accounts = base_accounts.dropna(subset=["account_first_login_at"]).copy()

print("Eligible accounts with first login:", len(base_accounts))
base_accounts.head()

Eligible accounts with first login: 1002


,account_id,account_name,industry,segment,estimated_annual_recurring_revenue,account_first_login_at
0,8f55adef-0abf-460f-9898-2cda579d9ba5,Senior Housing Properties Trust,Financial Services,Midmarket,4550000,2023-08-10 03:01:23
1,c5b92423-831e-457c-afdd-e517420672df,"Blackrock MuniEnhanced Fund, Inc.",Automotive,SMB,125000,2023-07-15 03:40:38
2,de9b63ea-04cc-43ec-b622-3969981f1d7c,"Maiden Holdings, Ltd.",Pharmaceuticals,Midmarket,650000,2023-07-04 08:58:32
3,a9867c25-fd83-4e1b-8551-3bff728b163d,"Carrols Restaurant Group, Inc.",Logistics,SMB,250000,2023-06-16 04:17:37
4,61830633-16a3-49ad-9ff7-37a7d72f3aa8,Dell Technologies Inc.,Education,Enterprise,17500000,2023-07-08 06:43:24


In [6]:
def build_baseline_features() -> pd.DataFrame:
    base = base_accounts.copy()

    # Users known at or before first login.
    users_with_login = users.merge(base[["account_id", "account_first_login_at"]], on="account_id", how="inner")
    users_pre = users_with_login[users_with_login["created_at"] <= users_with_login["account_first_login_at"]].copy()

    user_features = users_pre.groupby("account_id").agg(
        baseline_users=("user_id", "nunique"),
        baseline_marketing_opt_in_rate=("is_marketing_opted_in", "mean"),
        baseline_avg_experience_years=("experience_in_years", "mean"),
        baseline_max_experience_years=("experience_in_years", "max"),
        baseline_primary_job_title=("job_title", mode_or_unknown),
    ).reset_index()
    base = base.merge(user_features, on="account_id", how="left")

    # Marketing known before first login.
    leads_with_login = marketing_leads.merge(base[["account_id", "account_first_login_at"]], on="account_id", how="inner")
    leads_pre = leads_with_login[leads_with_login["created_at"] <= leads_with_login["account_first_login_at"]].copy()
    if not leads_pre.empty:
        lead_features = leads_pre.groupby("account_id").agg(
            baseline_leads=("lead_id", "nunique"),
            baseline_total_lead_cost=("lead_cost", "sum"),
            baseline_avg_lead_cost=("lead_cost", "mean"),
            baseline_primary_lead_source=("lead_source", mode_or_unknown),
            baseline_primary_campaign=("campaign_name", mode_or_unknown),
            baseline_primary_utm_medium=("utm_medium", mode_or_unknown),
            baseline_primary_sdr=("sdr", mode_or_unknown),
        ).reset_index()
        base = base.merge(lead_features, on="account_id", how="left")

    # Sales activity known before first login.
    activities_with_login = activities.merge(base[["account_id", "account_first_login_at"]], on="account_id", how="inner")
    activities_pre = activities_with_login[
        activities_with_login["activity_timestamp"] <= activities_with_login["account_first_login_at"]
    ].copy()
    if not activities_pre.empty:
        activity_features = activities_pre.groupby("account_id").agg(
            baseline_sales_activities=("activity_id", "count"),
            baseline_active_leads_touched=("lead_id", "nunique"),
            baseline_avg_call_duration_seconds=("call_duration_seconds", "mean"),
            baseline_last_sales_activity_at=("activity_timestamp", "max"),
        ).reset_index()
        base = base.merge(activity_features, on="account_id", how="left")
        base["baseline_days_since_last_sales_activity"] = (
            base["account_first_login_at"] - base["baseline_last_sales_activity_at"]
        ).dt.days
        base = base.drop(columns=["baseline_last_sales_activity_at"])

        for activity_type in activities_pre["activity_type"].dropna().unique():
            counts = (
                activities_pre[activities_pre["activity_type"].eq(activity_type)]
                .groupby("account_id")
                .size()
                .rename(f"baseline_activity_count__{activity_type}")
                .reset_index()
            )
            base = base.merge(counts, on="account_id", how="left")

    # Prior deals known before first login. Avoid using future final stage after first login.
    deals_with_login = deals.merge(base[["account_id", "account_first_login_at"]], on="account_id", how="inner")
    deals_pre = deals_with_login[deals_with_login["created_date"] <= deals_with_login["account_first_login_at"]].copy()
    if not deals_pre.empty:
        deal_features = deals_pre.groupby("account_id").agg(
            baseline_prior_deals=("deal_id", "nunique"),
            baseline_prior_pipeline_amount=("amount", "sum"),
            baseline_prior_avg_deal_amount=("amount", "mean"),
            baseline_prior_max_seats=("seats", "max"),
            baseline_primary_plan=("plan", mode_or_unknown),
        ).reset_index()
        base = base.merge(deal_features, on="account_id", how="left")

    # Geography known through leads before first login.
    geo_with_login = lead_geo.merge(base[["account_id", "account_first_login_at"]], on="account_id", how="inner")
    geo_pre = geo_with_login[geo_with_login["created_at"] <= geo_with_login["account_first_login_at"]].copy()
    if not geo_pre.empty:
        geo_features = geo_pre.groupby("account_id").agg(
            baseline_primary_continent=("continent", mode_or_unknown),
            baseline_primary_country=("country_name", mode_or_unknown),
            baseline_distinct_countries=("country_code", "nunique"),
        ).reset_index()
        base = base.merge(geo_features, on="account_id", how="left")

    base["first_login_month"] = base["account_first_login_at"].dt.to_period("M").astype(str)
    base["first_login_quarter"] = base["account_first_login_at"].dt.to_period("Q").astype(str)

    return base

baseline_df = build_baseline_features()
print(baseline_df.shape)
baseline_df.head()

(1002, 38)


,account_id,account_name,industry,segment,estimated_annual_recurring_revenue,account_first_login_at,baseline_users,baseline_marketing_opt_in_rate,baseline_avg_experience_years,baseline_max_experience_years,...,baseline_prior_deals,baseline_prior_pipeline_amount,baseline_prior_avg_deal_amount,baseline_prior_max_seats,baseline_primary_plan,baseline_primary_continent,baseline_primary_country,baseline_distinct_countries,first_login_month,first_login_quarter
0,8f55adef-0abf-460f-9898-2cda579d9ba5,Senior Housing Properties Trust,Financial Services,Midmarket,4550000,2023-08-10 03:01:23,1,1.0,6.0,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-08,2023Q3
1,c5b92423-831e-457c-afdd-e517420672df,"Blackrock MuniEnhanced Fund, Inc.",Automotive,SMB,125000,2023-07-15 03:40:38,2,0.5,3.5,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-07,2023Q3
2,de9b63ea-04cc-43ec-b622-3969981f1d7c,"Maiden Holdings, Ltd.",Pharmaceuticals,Midmarket,650000,2023-07-04 08:58:32,1,0.0,7.0,7,...,NaN,NaN,NaN,NaN,NaN,Asia,Japan,1.0,2023-07,2023Q3
3,a9867c25-fd83-4e1b-8551-3bff728b163d,"Carrols Restaurant Group, Inc.",Logistics,SMB,250000,2023-06-16 04:17:37,1,0.0,5.0,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-06,2023Q2
4,61830633-16a3-49ad-9ff7-37a7d72f3aa8,Dell Technologies Inc.,Education,Enterprise,17500000,2023-07-08 06:43:24,1,1.0,6.0,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023-07,2023Q3


## 6. Define Treatments and Outcomes

Each treatment is a feature adoption event inside an early time window after first login.

The outcome starts **after** the treatment window closes.

This avoids a common mistake: counting a conversion that happened before the treatment could plausibly have affected it.

In [7]:
@dataclass(frozen=True)
class TreatmentSpec:
    name: str
    event_name: str
    window_days: int
    business_question: str


treatments = [
    TreatmentSpec(
        name="workspace_created_7d",
        event_name="workspace_created",
        window_days=7,
        business_question="Does creating a workspace in the first 7 days increase 90-day conversion?",
    ),
    TreatmentSpec(
        name="report_generated_7d",
        event_name="report_generated",
        window_days=7,
        business_question="Does generating a report in the first 7 days increase 90-day conversion?",
    ),
    TreatmentSpec(
        name="workflow_created_14d",
        event_name="workflow_created",
        window_days=14,
        business_question="Does creating a workflow in the first 14 days increase 90-day conversion?",
    ),
    TreatmentSpec(
        name="api_call_made_14d",
        event_name="api_call_made",
        window_days=14,
        business_question="Does early API usage in the first 14 days increase 90-day conversion?",
    ),
]

In [8]:
def build_treatment_dataset(spec: TreatmentSpec) -> pd.DataFrame:
    df = baseline_df.copy()
    df["treatment_name"] = spec.name
    df["treatment_event"] = spec.event_name
    df["treatment_window_days"] = spec.window_days
    df["treatment_window_end"] = df["account_first_login_at"] + pd.to_timedelta(spec.window_days, unit="D")
    df["outcome_window_end"] = df["treatment_window_end"] + pd.to_timedelta(OUTCOME_HORIZON_DAYS, unit="D")

    # Treatment: did the account perform the event inside the treatment window?
    candidate_events = tracks[tracks["event_name"].eq(spec.event_name)].merge(
        df[["account_id", "account_first_login_at", "treatment_window_end"]],
        on="account_id",
        how="inner",
    )
    treated_accounts = candidate_events[
        (candidate_events["event_timestamp"] >= candidate_events["account_first_login_at"])
        & (candidate_events["event_timestamp"] <= candidate_events["treatment_window_end"])
    ]["account_id"].drop_duplicates()
    df["treatment"] = df["account_id"].isin(treated_accounts).astype(int)

    # Exclude accounts already won before treatment window closes.
    prior_wins = deals[deals["is_won"].eq(1)].merge(
        df[["account_id", "treatment_window_end"]], on="account_id", how="inner"
    )
    already_won_accounts = prior_wins[
        prior_wins["created_date"] <= prior_wins["treatment_window_end"]
    ]["account_id"].drop_duplicates()
    df = df[~df["account_id"].isin(already_won_accounts)].copy()

    # Outcome: won deal after treatment window closes and within the horizon.
    future_wins = deals[deals["is_won"].eq(1)].merge(
        df[["account_id", "treatment_window_end", "outcome_window_end"]], on="account_id", how="inner"
    )
    future_wins = future_wins[
        (future_wins["created_date"] > future_wins["treatment_window_end"])
        & (future_wins["created_date"] <= future_wins["outcome_window_end"])
    ]
    outcome = future_wins.groupby("account_id").agg(
        outcome=("deal_id", "nunique"),
        outcome_revenue=("amount", "sum"),
    ).reset_index()
    df = df.merge(outcome, on="account_id", how="left")
    df["outcome"] = (df["outcome"].fillna(0) > 0).astype(int)
    df["outcome_revenue"] = df["outcome_revenue"].fillna(0)

    # Keep only accounts whose full outcome window is observable in the data.
    max_observed_date = max(tracks["event_timestamp"].max(), deals["created_date"].max())
    df = df[df["outcome_window_end"] <= max_observed_date].copy()

    return df

causal_datasets = {spec.name: build_treatment_dataset(spec) for spec in treatments}

summary_rows = []
for spec in treatments:
    df = causal_datasets[spec.name]
    summary_rows.append({
        "treatment": spec.name,
        "event": spec.event_name,
        "window_days": spec.window_days,
        "accounts": len(df),
        "treated_rate": df["treatment"].mean(),
        "outcome_rate": df["outcome"].mean(),
        "treated_outcome_rate": df.loc[df["treatment"].eq(1), "outcome"].mean(),
        "control_outcome_rate": df.loc[df["treatment"].eq(0), "outcome"].mean(),
    })

treatment_summary = pd.DataFrame(summary_rows)
treatment_summary

,treatment,event,window_days,accounts,treated_rate,outcome_rate,treated_outcome_rate,control_outcome_rate
0,workspace_created_7d,workspace_created,7,949,0.059009,0.044257,0.125000,0.039194
1,report_generated_7d,report_generated,7,949,0.051633,0.044257,0.040816,0.044444
2,workflow_created_14d,workflow_created,14,945,0.023280,0.041270,0.045455,0.041170
3,api_call_made_14d,api_call_made,14,945,0.283598,0.041270,0.037313,0.042836


## 7. Preprocessing for Causal Models

We use only baseline covariates. Treatment and outcome columns are excluded.

Categorical features are one-hot encoded. Numeric features are imputed and scaled.

In [9]:
EXCLUDE_COLUMNS = {
    "account_id",
    "account_name",
    "account_first_login_at",
    "treatment_name",
    "treatment_event",
    "treatment_window_days",
    "treatment_window_end",
    "outcome_window_end",
    "treatment",
    "outcome",
    "outcome_revenue",
}


def feature_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if col not in EXCLUDE_COLUMNS]


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_one_hot_encoder()),
                ]),
                categorical_cols,
            ),
        ]
    )

## 8. Diagnostics: Naive Differences and Overlap

Before estimating causal effects, inspect two things:

1. **Naive difference**: How different are treated and untreated outcomes before adjustment?
2. **Overlap**: Are there comparable treated and untreated accounts with similar propensity scores?

If overlap is poor, the causal estimate is not credible for the full population.

In [10]:
def make_propensity_model(X: pd.DataFrame) -> Pipeline:
    # Do not use class_weight="balanced" here. A propensity score is a
    # probability of treatment, and class weighting distorts that probability.
    return Pipeline([
        ("preprocess", build_preprocessor(X)),
        ("model", LogisticRegression(max_iter=2000, C=0.5, random_state=RANDOM_STATE)),
    ])


def cross_fitted_propensity(df: pd.DataFrame, n_splits: int = 5) -> tuple[np.ndarray, float]:
    X = df[feature_columns(df)]
    treatment = df["treatment"].to_numpy()
    scores = np.full(len(df), np.nan)

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    for train_idx, valid_idx in splitter.split(X, treatment):
        model = make_propensity_model(X.iloc[train_idx])
        model.fit(X.iloc[train_idx], treatment[train_idx])
        scores[valid_idx] = model.predict_proba(X.iloc[valid_idx])[:, 1]

    # Numerical protection only. We later restrict causal interpretation to
    # common support instead of pretending extreme propensities are reliable.
    scores = np.clip(scores, 0.01, 0.99)
    auc = roc_auc_score(treatment, scores) if len(np.unique(treatment)) > 1 else np.nan
    return scores, auc


def overlap_diagnostics(df: pd.DataFrame, propensity: np.ndarray) -> dict[str, float]:
    treated = df["treatment"].to_numpy() == 1
    control = ~treated
    common_support = (propensity >= 0.05) & (propensity <= 0.95)
    return {
        "treated_propensity_mean": float(np.mean(propensity[treated])) if treated.any() else np.nan,
        "control_propensity_mean": float(np.mean(propensity[control])) if control.any() else np.nan,
        "treated_below_0_05": float(np.mean(propensity[treated] < 0.05)) if treated.any() else np.nan,
        "treated_above_0_95": float(np.mean(propensity[treated] > 0.95)) if treated.any() else np.nan,
        "control_below_0_05": float(np.mean(propensity[control] < 0.05)) if control.any() else np.nan,
        "control_above_0_95": float(np.mean(propensity[control] > 0.95)) if control.any() else np.nan,
        "common_support_rate": float(np.mean(common_support)),
    }


def naive_difference(df: pd.DataFrame) -> float:
    treated_rate = df.loc[df["treatment"].eq(1), "outcome"].mean()
    control_rate = df.loc[df["treatment"].eq(0), "outcome"].mean()
    return float(treated_rate - control_rate)


diagnostic_rows = []
propensity_scores_by_treatment = {}

for spec in treatments:
    df = causal_datasets[spec.name]
    propensity, p_auc = cross_fitted_propensity(df)
    propensity_scores_by_treatment[spec.name] = propensity
    row = {
        "treatment": spec.name,
        "accounts": len(df),
        "treated_accounts": int(df["treatment"].sum()),
        "control_accounts": int((1 - df["treatment"]).sum()),
        "outcome_events": int(df["outcome"].sum()),
        "treated_rate": df["treatment"].mean(),
        "outcome_rate": df["outcome"].mean(),
        "naive_difference": naive_difference(df),
        "cross_fitted_propensity_auc": p_auc,
    }
    row.update(overlap_diagnostics(df, propensity))
    diagnostic_rows.append(row)

diagnostics = pd.DataFrame(diagnostic_rows)
diagnostics


,treatment,accounts,treated_accounts,control_accounts,outcome_events,treated_rate,outcome_rate,naive_difference,cross_fitted_propensity_auc,treated_propensity_mean,control_propensity_mean,treated_below_0_05,treated_above_0_95,control_below_0_05,control_above_0_95,common_support_rate
0,workspace_created_7d,949,56,893,42,0.059009,0.044257,0.085806,0.654735,0.094237,0.060614,0.392857,0.0,0.677492,0.0,0.339305
1,report_generated_7d,949,49,900,42,0.051633,0.044257,-0.003628,0.419252,0.039599,0.052676,0.714286,0.0,0.661111,0.0,0.336143
2,workflow_created_14d,945,22,923,39,0.023280,0.041270,0.004284,0.493105,0.024744,0.026628,0.863636,0.0,0.903575,0.0,0.097354
3,api_call_made_14d,945,268,677,39,0.283598,0.041270,-0.005523,0.640446,0.350646,0.256067,0.003731,0.0,0.016248,0.0,0.987302


## 9. Estimators

We estimate each treatment effect with three approaches.

### 9.1 Naive Difference

```text
E[Y | T=1] - E[Y | T=0]
```

This is not causal unless treatment is as-if random.

### 9.2 Inverse Probability Weighted ATE

Weights control accounts by how likely they were to receive the treatment.

```text
ATE_IPW = mean(T * Y / e(X) - (1-T) * Y / (1-e(X)))
```

### 9.3 Doubly Robust AIPW

AIPW combines:

- propensity model: `P(T=1 | X)`
- outcome models: `E[Y | T=1, X]` and `E[Y | T=0, X]`

It is called doubly robust because it is consistent if either the propensity model or the outcome model is correctly specified.

### Reliability Correction: Repeated Cross-Fitting and Honest Uncertainty

The corrected implementation uses repeated five-fold cross-fitting, conservative regularized
nuisance models, explicit common-support restriction, and influence-function standard errors.
Averaging several shuffled fold assignments reduces split-specific instability while preserving
out-of-fold predictions.

These changes make confidence intervals more defensible; they do not force an interval to exclude
zero. Wide intervals remain the correct result when treatments, outcomes, or comparable controls
are scarce.


In [11]:
def ipw_ate(df: pd.DataFrame, propensity: np.ndarray) -> float:
    t = df["treatment"].to_numpy()
    y = df["outcome"].to_numpy()
    return float(np.mean(t * y / propensity - (1 - t) * y / (1 - propensity)))


def make_outcome_model(X: pd.DataFrame) -> Pipeline:
    return Pipeline([
        ("preprocess", build_preprocessor(X)),
        ("model", LogisticRegression(max_iter=2000, C=0.25, random_state=RANDOM_STATE)),
    ])


def constant_probability(y: np.ndarray, size: int) -> np.ndarray:
    probability = (float(np.sum(y)) + 1.0) / (len(y) + 2.0)
    return np.full(size, probability)


def one_cross_fit(
    df: pd.DataFrame,
    n_splits: int,
    random_state: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X = df[feature_columns(df)]
    t = df["treatment"].to_numpy()
    y = df["outcome"].to_numpy()

    propensity = np.full(len(df), np.nan)
    mu1 = np.full(len(df), np.nan)
    mu0 = np.full(len(df), np.nan)

    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, valid_idx in splitter.split(X, t):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        t_train, y_train = t[train_idx], y[train_idx]

        propensity_model = make_propensity_model(X_train)
        propensity_model.fit(X_train, t_train)
        propensity[valid_idx] = propensity_model.predict_proba(X_valid)[:, 1]

        for arm, destination in [(1, mu1), (0, mu0)]:
            arm_mask = t_train == arm
            arm_y = y_train[arm_mask]
            if len(np.unique(arm_y)) < 2:
                destination[valid_idx] = constant_probability(arm_y, len(valid_idx))
                continue

            outcome_model = make_outcome_model(X_train[arm_mask])
            outcome_model.fit(X_train[arm_mask], arm_y)
            destination[valid_idx] = outcome_model.predict_proba(X_valid)[:, 1]

    return propensity, mu1, mu0


def cross_fitted_nuisance_predictions(
    df: pd.DataFrame,
    n_splits: int = 5,
    repeats: int = CROSS_FIT_REPEATS,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Average honest out-of-fold nuisance predictions across repeated splits."""
    treatment_counts = df["treatment"].value_counts()
    smallest_arm = int(treatment_counts.min())
    safe_splits = max(2, min(n_splits, smallest_arm))

    repeated = [
        one_cross_fit(df, safe_splits, RANDOM_STATE + 1009 * repeat)
        for repeat in range(repeats)
    ]
    propensity = np.mean([item[0] for item in repeated], axis=0)
    mu1 = np.mean([item[1] for item in repeated], axis=0)
    mu0 = np.mean([item[2] for item in repeated], axis=0)

    propensity = np.clip(propensity, 0.01, 0.99)
    mu1 = np.clip(mu1, 1e-6, 1 - 1e-6)
    mu0 = np.clip(mu0, 1e-6, 1 - 1e-6)
    return propensity, mu1, mu0


def aipw_scores(
    df: pd.DataFrame,
    propensity: np.ndarray,
    mu1: np.ndarray,
    mu0: np.ndarray,
) -> np.ndarray:
    t = df["treatment"].to_numpy()
    y = df["outcome"].to_numpy()
    return mu1 - mu0 + t * (y - mu1) / propensity - (1 - t) * (y - mu0) / (1 - propensity)


def influence_function_ci(
    scores: np.ndarray,
    confidence_level: float = CONFIDENCE_LEVEL,
) -> tuple[float, float, float, float]:
    scores = np.asarray(scores, dtype=float)
    scores = scores[np.isfinite(scores)]
    estimate = float(np.mean(scores))
    standard_error = float(np.std(scores, ddof=1) / np.sqrt(len(scores)))
    # 1.96 is the two-sided 95% standard-normal critical value.
    critical_value = 1.96 if confidence_level == 0.95 else 1.96
    lower = estimate - critical_value * standard_error
    upper = estimate + critical_value * standard_error
    return estimate, standard_error, lower, upper


def estimate_effects(df: pd.DataFrame) -> dict[str, Any]:
    propensity, mu1, mu0 = cross_fitted_nuisance_predictions(df)
    cate = mu1 - mu0
    factual_probability = np.where(df["treatment"].to_numpy() == 1, mu1, mu0)
    common_support = (propensity >= 0.05) & (propensity <= 0.95)
    scores = aipw_scores(df, propensity, mu1, mu0)

    aipw_ate, aipw_se, ci_lower, ci_upper = influence_function_ci(scores)
    if common_support.sum() >= 2:
        support_ate, support_se, support_lower, support_upper = influence_function_ci(
            scores[common_support]
        )
    else:
        support_ate = support_se = support_lower = support_upper = np.nan

    treatment = df["treatment"].to_numpy()
    return {
        "naive_difference": naive_difference(df),
        "ipw_ate": ipw_ate(df, propensity),
        "aipw_ate": aipw_ate,
        "aipw_standard_error": aipw_se,
        "ci_lower_95": ci_lower,
        "ci_upper_95": ci_upper,
        "aipw_ate_common_support": support_ate,
        "aipw_common_support_standard_error": support_se,
        "common_support_ci_lower_95": support_lower,
        "common_support_ci_upper_95": support_upper,
        "treated_rate": df.loc[df["treatment"].eq(1), "outcome"].mean(),
        "control_rate": df.loc[df["treatment"].eq(0), "outcome"].mean(),
        "propensity": propensity,
        "mu1": mu1,
        "mu0": mu0,
        "cate": cate,
        "common_support": common_support,
        "common_support_accounts": int(common_support.sum()),
        "common_support_treated": int(np.sum(common_support & (treatment == 1))),
        "common_support_control": int(np.sum(common_support & (treatment == 0))),
        "factual_brier_score": brier_score_loss(df["outcome"], factual_probability),
        "mu1_p99": float(np.quantile(mu1, 0.99)),
        "mu0_p99": float(np.quantile(mu0, 0.99)),
        "cate_p99": float(np.quantile(cate, 0.99)),
    }


effect_objects = {}
effect_rows = []

for spec in treatments:
    df = causal_datasets[spec.name]
    effects = estimate_effects(df)
    effect_objects[spec.name] = effects
    propensity_scores_by_treatment[spec.name] = effects["propensity"]
    effect_rows.append({
        "treatment": spec.name,
        "question": spec.business_question,
        "accounts": len(df),
        "treated_accounts": int(df["treatment"].sum()),
        "control_accounts": int((1 - df["treatment"]).sum()),
        "treated_rate": df["treatment"].mean(),
        "outcome_rate": df["outcome"].mean(),
        "treated_outcome_rate": effects["treated_rate"],
        "control_outcome_rate": effects["control_rate"],
        "naive_difference": effects["naive_difference"],
        "ipw_ate": effects["ipw_ate"],
        "aipw_ate": effects["aipw_ate"],
        "aipw_standard_error": effects["aipw_standard_error"],
        "ci_lower_95": effects["ci_lower_95"],
        "ci_upper_95": effects["ci_upper_95"],
        "aipw_ate_common_support": effects["aipw_ate_common_support"],
        "aipw_common_support_standard_error": effects["aipw_common_support_standard_error"],
        "common_support_ci_lower_95": effects["common_support_ci_lower_95"],
        "common_support_ci_upper_95": effects["common_support_ci_upper_95"],
        "common_support_rate": effects["common_support"].mean(),
        "common_support_accounts": effects["common_support_accounts"],
        "common_support_treated": effects["common_support_treated"],
        "common_support_control": effects["common_support_control"],
        "factual_brier_score": effects["factual_brier_score"],
        "mu1_p99": effects["mu1_p99"],
        "mu0_p99": effects["mu0_p99"],
        "cate_p99": effects["cate_p99"],
    })

effect_summary = pd.DataFrame(effect_rows).sort_values(
    ["common_support_rate", "aipw_ate_common_support"],
    ascending=[False, False],
)
effect_summary


,treatment,question,accounts,treated_accounts,control_accounts,treated_rate,outcome_rate,treated_outcome_rate,control_outcome_rate,naive_difference,...,common_support_ci_lower_95,common_support_ci_upper_95,common_support_rate,common_support_accounts,common_support_treated,common_support_control,factual_brier_score,mu1_p99,mu0_p99,cate_p99
3,api_call_made_14d,Does early API usage in the first 14 days incr...,945,268,677,0.283598,0.041270,0.037313,0.042836,-0.005523,...,-0.044325,0.047165,0.992593,938,267,671,0.041794,0.173662,0.214126,0.155237
0,workspace_created_7d,Does creating a workspace in the first 7 days ...,949,56,893,0.059009,0.044257,0.125000,0.039194,0.085806,...,-0.028821,0.249380,0.378293,359,36,323,0.044195,0.520031,0.173629,0.475898
1,report_generated_7d,Does generating a report in the first 7 days i...,949,49,900,0.051633,0.044257,0.040816,0.044444,-0.003628,...,-0.042947,0.005887,0.350896,333,18,315,0.043600,0.457854,0.228365,0.444459
2,workflow_created_14d,Does creating a workflow in the first 14 days ...,945,22,923,0.023280,0.041270,0.045455,0.041170,0.004284,...,-0.025829,0.056385,0.091005,86,3,83,0.041482,0.087639,0.175410,0.061157


## 10. Confidence Intervals and Precision Diagnostics

The primary intervals use the empirical variance of the cross-fitted AIPW influence scores.
We report intervals for both the full population and the common-support population. The latter
is the safer decision target when treatment is rare or propensity overlap is weak.


In [12]:
causal_results = effect_summary.copy()

precision_columns = [
    "treatment",
    "treated_accounts",
    "aipw_ate",
    "aipw_standard_error",
    "ci_lower_95",
    "ci_upper_95",
    "common_support_rate",
    "common_support_treated",
    "aipw_ate_common_support",
    "aipw_common_support_standard_error",
    "common_support_ci_lower_95",
    "common_support_ci_upper_95",
]
causal_results[precision_columns]


,treatment,treated_accounts,aipw_ate,aipw_standard_error,ci_lower_95,ci_upper_95,common_support_rate,common_support_treated,aipw_ate_common_support,aipw_common_support_standard_error,common_support_ci_lower_95,common_support_ci_upper_95
3,api_call_made_14d,268,-0.001619,0.023381,-0.047445,0.044208,0.992593,267,0.001420,0.023339,-0.044325,0.047165
0,workspace_created_7d,56,0.106645,0.118166,-0.124960,0.338250,0.378293,36,0.110280,0.070970,-0.028821,0.249380
1,report_generated_7d,49,0.087835,0.124528,-0.156240,0.331909,0.350896,18,-0.018530,0.012458,-0.042947,0.005887
2,workflow_created_14d,22,-0.007103,0.067802,-0.139994,0.125788,0.091005,3,0.015278,0.020973,-0.025829,0.056385


## 11. Heterogeneous Treatment Effects

The average effect is useful, but product decisions require knowing **who benefits**.

We segment treatment effects by:

- Account segment
- Industry
- ARR band
- Baseline user count band
- Marketing source

This helps identify where an intervention is most likely to improve customer experience and conversion.

In [13]:
def add_effect_segments(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["arr_band"] = pd.qcut(
        out["estimated_annual_recurring_revenue"].rank(method="first"),
        q=4,
        labels=["Low ARR", "Mid ARR", "High ARR", "Very high ARR"],
    )
    out["baseline_user_band"] = pd.cut(
        out["baseline_users"].fillna(0),
        bins=[-1, 0, 1, 3, 10, np.inf],
        labels=["No users", "1 user", "2-3 users", "4-10 users", "10+ users"],
    )
    return out


def segment_effect_table(
    treatment_name: str,
    segment_col: str,
    min_accounts: int = 50,
    min_support_treated: int = 10,
    min_support_control: int = 25,
) -> pd.DataFrame:
    df = add_effect_segments(causal_datasets[treatment_name]).copy()
    effects = effect_objects[treatment_name]
    df["common_support"] = effects["common_support"]
    df["aipw_score"] = aipw_scores(
        df,
        effects["propensity"],
        effects["mu1"],
        effects["mu0"],
    )

    rows = []
    for value, group in df.groupby(segment_col, dropna=False, observed=True):
        support_group = group[group["common_support"]].copy()
        support_treated = int(support_group["treatment"].sum())
        support_control = int((1 - support_group["treatment"]).sum())

        if (
            len(group) < min_accounts
            or support_treated < min_support_treated
            or support_control < min_support_control
        ):
            continue

        estimate, standard_error, lower, upper = influence_function_ci(
            support_group["aipw_score"].to_numpy()
        )
        rows.append({
            "treatment": treatment_name,
            "segment_type": segment_col,
            "segment_value": str(value),
            "accounts": len(group),
            "support_accounts": len(support_group),
            "support_rate": len(support_group) / len(group),
            "support_treated": support_treated,
            "support_control": support_control,
            "treated_rate": group["treatment"].mean(),
            "outcome_rate": group["outcome"].mean(),
            "naive_difference": naive_difference(group),
            "avg_cate": estimate,
            "standard_error": standard_error,
            "ci_lower_95": lower,
            "ci_upper_95": upper,
            "interval_excludes_zero": bool(lower > 0 or upper < 0),
        })
    return pd.DataFrame(rows)


segment_tables = []
for spec in treatments:
    for segment_col in [
        "segment",
        "industry",
        "arr_band",
        "baseline_user_band",
        "baseline_primary_lead_source",
    ]:
        if segment_col in add_effect_segments(causal_datasets[spec.name]).columns:
            table = segment_effect_table(spec.name, segment_col)
            if not table.empty:
                segment_tables.append(table)

if segment_tables:
    hte_summary = pd.concat(segment_tables, ignore_index=True).sort_values(
        ["ci_lower_95", "avg_cate"],
        ascending=False,
    )
else:
    hte_summary = pd.DataFrame(columns=[
        "treatment",
        "segment_type",
        "segment_value",
        "accounts",
        "support_accounts",
        "support_rate",
        "support_treated",
        "support_control",
        "treated_rate",
        "outcome_rate",
        "naive_difference",
        "avg_cate",
        "standard_error",
        "ci_lower_95",
        "ci_upper_95",
        "interval_excludes_zero",
    ])

hte_summary.head(30)


,treatment,segment_type,segment_value,accounts,support_accounts,support_rate,support_treated,support_control,treated_rate,outcome_rate,naive_difference,avg_cate,standard_error,ci_lower_95,ci_upper_95,interval_excludes_zero
2,workspace_created_7d,baseline_user_band,1 user,909,332,0.365237,31,301,0.056106,0.044004,0.098793,0.138254,0.074639,-0.008038,0.284547,False
17,api_call_made_14d,industry,Media and Entertainment,56,56,1.000000,17,39,0.303571,0.017857,0.058824,0.023714,0.030531,-0.036127,0.083555,False
6,report_generated_7d,baseline_user_band,1 user,909,313,0.344334,16,297,0.050605,0.044004,-0.000554,-0.015592,0.012621,-0.040329,0.009145,False
23,api_call_made_14d,arr_band,High ARR,236,236,1.000000,71,165,0.300847,0.046610,0.034059,0.035513,0.039477,-0.041863,0.112888,False
9,api_call_made_14d,segment,Midmarket,278,276,0.992806,81,195,0.291367,0.043165,0.026195,0.030351,0.037190,-0.042541,0.103243,False
25,api_call_made_14d,baseline_user_band,1 user,905,898,0.992265,243,655,0.269613,0.040884,-0.011086,-0.002994,0.024083,-0.050197,0.044208,False
7,report_generated_7d,baseline_primary_lead_source,nan,569,205,0.360281,12,193,0.054482,0.040422,-0.008634,-0.025563,0.015905,-0.056737,0.005610,False
4,report_generated_7d,segment,Midmarket,278,136,0.489209,10,126,0.068345,0.043165,-0.046332,-0.021546,0.019403,-0.059576,0.016484,False
30,api_call_made_14d,baseline_primary_lead_source,nan,568,567,0.998239,137,430,0.241197,0.040493,-0.024506,-0.008467,0.029683,-0.066645,0.049710,False
10,api_call_made_14d,segment,SMB,522,517,0.990421,146,371,0.281609,0.040230,-0.027592,-0.024928,0.022982,-0.069972,0.020116,False


## 12. Account-Level Uplift and Intervention Candidates

Now we translate causal estimates into product and GTM action.

For each account, we estimate:

```text
CATE = predicted outcome if treated - predicted outcome if untreated
```

Accounts with high CATE are likely to benefit from an intervention that encourages the treatment.

Important distinction:

- High propensity to convert = likely buyer
- High uplift = likely to be influenced by intervention

A good product intervention targets high-uplift accounts, not just high-propensity accounts.

In [14]:
def intervention_candidates(treatment_name: str, top_n: int = 50) -> pd.DataFrame:
    df = causal_datasets[treatment_name].copy()
    effects = effect_objects[treatment_name]
    df["propensity_score"] = effects["propensity"]
    df["predicted_conversion_if_treated"] = effects["mu1"]
    df["predicted_conversion_if_control"] = effects["mu0"]
    df["estimated_uplift"] = effects["cate"]
    df["common_support"] = effects["common_support"]

    # Only rank untreated accounts where treated and untreated analogues exist.
    # This avoids counterfactual extrapolation into unsupported populations.
    candidates = df[
        df["treatment"].eq(0)
        & df["common_support"]
        & df["propensity_score"].between(0.05, 0.95)
    ].copy()

    # Extreme predictions are a diagnostic, not a reason to rank an account.
    # With the cross-fitted regularized model, values near 1 should be rare.
    candidates["probability_reliability"] = np.select(
        [
            candidates["predicted_conversion_if_treated"].between(0.01, 0.60)
            & candidates["predicted_conversion_if_control"].between(0.001, 0.50),
            candidates["predicted_conversion_if_treated"].between(0.0, 0.80),
        ],
        ["higher", "moderate"],
        default="review",
    )

    candidates = candidates.sort_values(
        ["estimated_uplift", "probability_reliability"],
        ascending=[False, True],
    ).head(top_n)

    return candidates[[
        "account_id",
        "account_name",
        "industry",
        "segment",
        "estimated_annual_recurring_revenue",
        "treatment_name",
        "propensity_score",
        "predicted_conversion_if_treated",
        "predicted_conversion_if_control",
        "estimated_uplift",
        "probability_reliability",
        "outcome",
        "outcome_revenue",
    ]]


candidate_tables = []
for spec in treatments:
    part = intervention_candidates(spec.name, top_n=25)
    candidate_tables.append(part)

intervention_candidates_df = pd.concat(candidate_tables, ignore_index=True).sort_values(
    "estimated_uplift", ascending=False
)

print("Counterfactual probability diagnostics after correction:")
display(effect_summary[[
    "treatment",
    "outcome_rate",
    "mu1_p99",
    "mu0_p99",
    "cate_p99",
    "factual_brier_score",
    "common_support_rate",
]])

intervention_candidates_df.head(30)


Counterfactual probability diagnostics after correction:


,treatment,outcome_rate,mu1_p99,mu0_p99,cate_p99,factual_brier_score,common_support_rate
3,api_call_made_14d,0.041270,0.173662,0.214126,0.155237,0.041794,0.992593
0,workspace_created_7d,0.044257,0.520031,0.173629,0.475898,0.044195,0.378293
1,report_generated_7d,0.044257,0.457854,0.228365,0.444459,0.043600,0.350896
2,workflow_created_14d,0.041270,0.087639,0.175410,0.061157,0.041482,0.091005


,account_id,account_name,industry,segment,estimated_annual_recurring_revenue,treatment_name,propensity_score,predicted_conversion_if_treated,predicted_conversion_if_control,estimated_uplift,probability_reliability,outcome,outcome_revenue
0,4dcaef75-3ef3-4534-9ba2-ef42768c31e9,"Kforce, Inc.",Technology,SMB,375000,workspace_created_7d,0.174375,0.755417,0.050731,0.704686,moderate,0,0.0
1,bc3a706c-b887-4570-a5e7-db5809046ba1,Great Plains Energy Inc,Energy,Enterprise,41200000,workspace_created_7d,0.077866,0.641579,0.004005,0.637574,moderate,0,0.0
2,f10e28b8-c0e2-44ce-a2c1-ecd83ad387d8,China Xiniya Fashion Limited,Media and Entertainment,Enterprise,6300000,workspace_created_7d,0.133177,0.631833,0.014574,0.617259,moderate,0,0.0
3,912d0f9c-3479-41b9-b7b1-67d2aae4cb2e,National Steel Company,Telecommunications,Midmarket,4200000,workspace_created_7d,0.231672,0.671712,0.067413,0.604298,moderate,0,0.0
4,1230c262-7418-4b11-8135-f5eca7bd00df,Vanguard Russell 3000 ETF,Healthcare,SMB,450000,workspace_created_7d,0.130869,0.610416,0.051743,0.558673,moderate,0,0.0
75,40e41ef3-5f78-4fe6-8576-b05b8258edd5,DMC Global Inc.,Energy,Enterprise,49900000,api_call_made_14d,0.548153,0.555567,0.025370,0.530197,higher,0,0.0
5,79ebe464-035b-40a0-a876-6fcf9cc37c08,DCP Midstream LP,Automotive,Midmarket,3550000,workspace_created_7d,0.110820,0.572862,0.044058,0.528804,higher,0,0.0
6,b2d20266-6339-409c-b2ce-f56bdef14275,IHS Markit Ltd.,Transportation,SMB,125000,workspace_created_7d,0.052393,0.491826,0.007739,0.484087,higher,0,0.0
7,ac96e7ae-5e33-4263-be01-9b7126202ea7,Gulfport Energy Corporation,Automotive,Midmarket,2750000,workspace_created_7d,0.054149,0.498318,0.031291,0.467026,higher,0,0.0
8,1cbfc176-b093-49af-8d72-e7938be77277,"Frankly, Inc.",Construction,SMB,500000,workspace_created_7d,0.090423,0.487480,0.020886,0.466595,higher,0,0.0


## 13. Product and Customer Experience Interpretation

The causal results should be translated into product decisions, not left as model output.

For each treatment, we classify the recommendation:

- **Scale**: positive effect and confidence interval mostly above zero
- **Experiment**: promising but uncertain
- **Do not prioritize**: weak or negative estimated effect
- **Segmented rollout**: average effect is modest, but some segments show strong uplift

In [15]:
def recommendation_from_effect(row: pd.Series) -> str:
    support = row["common_support_rate"]
    support_treated = row["common_support_treated"]
    estimate = row["aipw_ate_common_support"]
    lower = row["common_support_ci_lower_95"]
    upper = row["common_support_ci_upper_95"]

    if support < 0.25 or support_treated < 20:
        return "Insufficient overlap / treated sample"
    if pd.notna(lower) and lower > 0:
        return "Promising evidence — confirm with experiment"
    if pd.notna(upper) and upper < 0:
        return "Do not prioritize broadly"
    if pd.notna(estimate) and estimate > 0:
        return "Exploratory experiment only — interval includes zero"
    return "Inconclusive — collect more data"


def product_hypothesis(treatment: str) -> str:
    if treatment == "workspace_created_7d":
        return "Improve onboarding toward workspace creation; test guided setup and workspace templates."
    if treatment == "report_generated_7d":
        return "Reduce time-to-first-report; test report templates, sample data, and in-app prompts."
    if treatment == "workflow_created_14d":
        return "Help users operationalize value; test workflow recommendations after dashboard/report use."
    if treatment == "api_call_made_14d":
        return "Route technical accounts to API onboarding, docs, and solutions-engineer support."
    return "Design a targeted product intervention."


causal_results["recommendation"] = causal_results.apply(recommendation_from_effect, axis=1)
causal_results["product_hypothesis"] = causal_results["treatment"].apply(product_hypothesis)
causal_results[[
    "treatment",
    "aipw_ate",
    "ci_lower_95",
    "aipw_ate_common_support",
    "common_support_ci_lower_95",
    "common_support_ci_upper_95",
    "common_support_rate",
    "common_support_treated",
    "recommendation",
]]


,treatment,aipw_ate,ci_lower_95,aipw_ate_common_support,common_support_ci_lower_95,common_support_ci_upper_95,common_support_rate,common_support_treated,recommendation
3,api_call_made_14d,-0.001619,-0.047445,0.001420,-0.044325,0.047165,0.992593,267,Exploratory experiment only — interval include...
0,workspace_created_7d,0.106645,-0.124960,0.110280,-0.028821,0.249380,0.378293,36,Exploratory experiment only — interval include...
1,report_generated_7d,0.087835,-0.156240,-0.018530,-0.042947,0.005887,0.350896,18,Insufficient overlap / treated sample
2,workflow_created_14d,-0.007103,-0.139994,0.015278,-0.025829,0.056385,0.091005,3,Insufficient overlap / treated sample


## 14. Experiment Roadmap

Observational causal inference should feed experiments.

For the strongest treatment, design an A/B test:

### Example Experiment

```text
Population:
  Accounts that logged in but did not create a workspace in first 24 hours

Treatment:
  Guided workspace setup experience + template recommendation

Control:
  Existing onboarding

Primary outcome:
  Workspace created within 7 days

Secondary outcomes:
  Report generated within 14 days
  Account activation within 30 days
  Won deal within 90 days

Guardrails:
  Support tickets / negative feedback
  Unsubscribe rate
  Product errors
```

### Why Experiment After Causal Inference?

Causal inference narrows the opportunity space. Experiments validate the actual product intervention.

In [16]:
eligible_experiments = causal_results[
    causal_results["common_support_rate"].ge(0.25)
    & causal_results["common_support_treated"].ge(20)
].copy()

if eligible_experiments.empty:
    best_treatment = causal_results.sort_values(
        ["common_support_rate", "common_support_treated"],
        ascending=False,
    ).iloc[0]
else:
    # Rank experimental hypotheses by the lower confidence bound in common support,
    # then by the support-restricted point estimate. This avoids choosing a treatment
    # merely because an unstable full-population point estimate is fractionally larger.
    best_treatment = eligible_experiments.sort_values(
        ["common_support_ci_lower_95", "aipw_ate_common_support"],
        ascending=False,
    ).iloc[0]

experiment_brief = pd.DataFrame([
    {
        "field": "Treatment to test",
        "recommendation": best_treatment["treatment"],
    },
    {
        "field": "Why this treatment",
        "recommendation": best_treatment["product_hypothesis"],
    },
    {
        "field": "Estimated AIPW effect",
        "recommendation": f"{best_treatment['aipw_ate_common_support']:.4f}",
    },
    {
        "field": "95% CI in common support",
        "recommendation": (
            f"{best_treatment['common_support_ci_lower_95']:.4f} to "
            f"{best_treatment['common_support_ci_upper_95']:.4f}"
        ),
    },
    {
        "field": "Common support",
        "recommendation": f"{best_treatment['common_support_rate']:.1%}",
    },
    {
        "field": "Recommended next step",
        "recommendation": best_treatment["recommendation"],
    },
    {
        "field": "Primary proximal outcome",
        "recommendation": "Feature adoption within treatment window",
    },
    {
        "field": "Primary business outcome",
        "recommendation": "90-day Won / Closed Won conversion",
    },
])
experiment_brief


,field,recommendation
0,Treatment to test,workspace_created_7d
1,Why this treatment,Improve onboarding toward workspace creation; ...
2,Estimated AIPW effect,0.1103
3,95% CI in common support,-0.0288 to 0.2494
4,Common support,37.8%
5,Recommended next step,Exploratory experiment only — interval include...
6,Primary proximal outcome,Feature adoption within treatment window
7,Primary business outcome,90-day Won / Closed Won conversion


## 15. Save Outputs

These artifacts can be used by the website, Snowflake publish step, or a product review deck.

In [17]:
OUTPUT_DIR = Path("causal_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

treatment_summary.to_csv(OUTPUT_DIR / "treatment_summary.csv", index=False)
diagnostics.to_csv(OUTPUT_DIR / "propensity_overlap_diagnostics.csv", index=False)
causal_results.to_csv(OUTPUT_DIR / "causal_effect_estimates.csv", index=False)
hte_summary.to_csv(OUTPUT_DIR / "heterogeneous_treatment_effects.csv", index=False)
intervention_candidates_df.to_csv(OUTPUT_DIR / "intervention_candidates.csv", index=False)
experiment_brief.to_csv(OUTPUT_DIR / "experiment_brief.csv", index=False)

print("Saved causal outputs to", OUTPUT_DIR.resolve())

Saved causal outputs to /Users/naghmeh6/agentic_ai/ABtesting/causal_outputs


## 16. Director-Level Readout Template

Use this structure for the final product/data science readout.

### 1. What We Tested

We tested whether early adoption of key product features increases 90-day account conversion.

### 2. What We Found

Report:

- AIPW effect estimate
- 95% confidence interval
- treatment adoption rate
- conversion rate among treated and controls
- segments with strongest uplift

### 3. What It Means For Product

Translate effect into product opportunity:

- onboarding improvement
- feature discoverability
- workflow simplification
- API enablement
- report template guidance

### 4. What It Means For GTM

Use uplift to decide who gets which intervention:

- SDR outreach
- AE call
- Solutions engineer support
- Lifecycle email
- In-app guide

### 5. What We Should Experiment On Next

Pick the treatment with the strongest causal signal and design a controlled experiment.

### 6. What We Should Monitor

- treatment adoption
- downstream activation
- 90-day conversion
- segment-level effects
- unintended friction
- score/uplift drift over time

### Final Principle

Do not use this analysis to declare permanent truth. Use it to identify the highest-value product experiment and the highest-uplift customer segments.